# IMDB review dataset cleaning

This script is used to clean the IMDB review dataset.
Let's import the Hugging Face dataset library called stanfordnlp/imdb

In [41]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb", split="train")

## Convert dataset into pandas
Now we will convert the dataset into a pandas dataframe for easier manipulation and cleaning.

In [42]:
df = dataset.to_pandas()

## Let's analyze the dataset
Now that we have the Hugging Face dataset loaded and converted into a pandas dataframe, let's analyze the dataset to understand its structure and content. We will look at the first few rows of the dataframe, check for missing values, and get a summary of the data.

In [43]:
print("First 5 rows of the dataset ")
print("-----------------------------")
print(df.head())
print()
print()

print("Summary of the dataset ")
print("------------------------")
print(df.info())
print()
print()

print("Distribution of labels in the dataset ")
print("----------------------------------")
print(df["label"].value_counts())


First 5 rows of the dataset 
-----------------------------
                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t...      0
3  This film was probably inspired by Godard's Ma...      0
4  Oh, brother...after hearing about this ridicul...      0


Summary of the dataset 
------------------------
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    25000 non-null  str  
 1   label   25000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 32.0 MB
None


Distribution of labels in the dataset 
----------------------------------
label
0    12500
1    12500
Name: count, dtype: int64


## Check for duplicates or missing text
Now we will check for duplicates or missing text in the reviews. We will remove any duplicate reviews and drop any rows with missing text.

In [44]:
print("Check for missing or empty text")
print(df["text"].isna().sum())
print((df["text"].str.strip() == "").sum())

Check for missing or empty text
0
0


now let's check for duplicates

In [45]:
print(df.duplicated(subset=["text"]).sum())

96


We found 96 duplicate reviews in the dataset. We will remove these duplicates to ensure that our model is trained on unique reviews only.

In [46]:
df = df.dropna(subset=["text"])
df = df[df["text"].str.strip() != ""]

## Let's add useful columns for analysis
Now that we have cleaned the dataset, let's add some useful columns for analysis. We will add

In [47]:
df["char_length"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()
print("Now our dataframe looks like this")
print(df.head())

Now our dataframe looks like this
                                                text  label  char_length  \
0  I rented I AM CURIOUS-YELLOW from my video sto...      0         1640   
1  "I Am Curious: Yellow" is a risible and preten...      0         1294   
2  If only to avoid making this type of film in t...      0          528   
3  This film was probably inspired by Godard's Ma...      0          706   
4  Oh, brother...after hearing about this ridicul...      0         1814   

   word_count  
0         288  
1         214  
2          93  
3         118  
4         311  


## Analyze the distribution of review lengths
Now that we have added the character length and word count columns, let's analyze the distribution of review

In [49]:
print("Let's look for very short rows")
df["word_count"].describe()
df[df["word_count"] < 5][["text", "label"]].head(10)

Let's look for very short rows


,text,label


We found no short rows, so we can assume that they are all valid reviews.

Let's map the labels to readable names
IMDB labels are usually: 0 = negative, 1 = positive

In [50]:
df["label_name"] = df["label"].map({0: "negative", 1: "positive"})
df[["label", "label_name"]].head()

,label,label_name
0,0,negative
1,0,negative
2,0,negative
3,0,negative
4,0,negative


## Lets do a few practical analyses on the dataset
Average review length by sentiment

In [52]:
review_length_by_sentiment = df.groupby("label_name")["word_count"].mean()
print("Average review length by sentiment")
print(review_length_by_sentiment)

Average review length by sentiment
label_name
negative    230.86784
positive    236.70656
Name: word_count, dtype: float64


Count of reviews by sentiment

In [53]:
df["label_name"].value_counts()

label_name
negative    12500
positive    12500
Name: count, dtype: int64

Longest reviews

In [54]:
df.sort_values("word_count", ascending=False)[["text", "label_name", "word_count"]].head(10)

,text,label_name,word_count
13756,Match 1: Tag Team Table Match Bubba Ray and Sp...,positive,2470
22551,Titanic directed by James Cameron presents a f...,positive,1839
16948,**Attention Spoilers**<br /><br />First of all...,positive,1830
16487,By now you've probably heard a bit about the n...,positive,1723
15085,*!!- SPOILERS - !!*<br /><br />Before I begin ...,positive,1601
20010,Warning: Does contain spoilers.<br /><br />Ope...,positive,1527
2037,Some have praised _Atlantis:_The_Lost_Empire_ ...,negative,1522
2038,Some have praised -Atlantis:-The Lost Empire- ...,negative,1475
16721,Polish film maker Walerian Borowczyk's La Bête...,positive,1398
7761,"*** Warning - this review contains ""plot spoil...",negative,1376


Shortest reviews

In [55]:
df.sort_values("word_count")[["text", "label_name", "word_count"]].head(10)

,text,label_name,word_count
10925,This movie is terrible but it has some good ef...,negative,10
2404,I wouldn't rent this one even on dollar rental...,negative,10
373,You'd better choose Paul Verhoeven's even if y...,negative,11
17069,Adrian Pasdar is excellent is this film. He ma...,positive,12
12114,Ming The Merciless does a little Bardwork and ...,negative,12
789,"Long, boring, blasphemous. Never have I been s...",negative,14
1629,"Comment this movie is impossible. Is terrible,...",negative,15
7220,"no comment - stupid movie, acting average or w...",negative,17
15022,This is the definitive movie version of Hamlet...,positive,17
5296,"A rating of ""1"" does not begin to express how ...",negative,18
